# Problem Set 5B - Solutions

**Topic: NumPy.** Total: 10 marks over 5 questions. **Submitted on Yaksh.**

Instructor / TA copy. These are the reference answers that the test cases in
`Assignment_5B.yaml` were checked against, together with the worked examples
each question quotes. Every question carries three PyHamcrest assertion test
cases and one structural hook test case on the portal, half a mark each.


## Question 1. Standardise the columns of an array

Write a function `standardize(a)` that takes a two dimensional NumPy array of
floats and returns an array of the **same shape** in which every column has
zero mean and unit standard deviation.

For each column, subtract the mean of that column and divide by the standard
deviation of that column. Use the default `np.std`, that is `ddof=0`.

Use the `axis` argument together with broadcasting. Do not write a `for` or
`while` loop and do not use a list comprehension. You may assume that no column
is constant.

For example, for

```python
a = np.array([[1.0, 2.0],
              [3.0, 6.0],
              [5.0, 10.0]])
```

`standardize(a)` returns

```
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
```


In [1]:
import numpy as np


def standardize(a):
    return (a - a.mean(axis=0)) / a.std(axis=0)


# Check against the example in the question.
a = np.array([[1.0, 2.0], [3.0, 6.0], [5.0, 10.0]])
result = standardize(a)
print(result)
print("shape       :", result.shape, "(must match", a.shape, ")")
print("column means:", result.mean(axis=0), "(must be 0)")
print("column stds :", result.std(axis=0), "(must be 1)")

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
shape       : (3, 2) (must match (3, 2) )
column means: [0. 0.] (must be 0)
column stds : [1. 1.] (must be 1)


## Question 2. Pairwise distance matrix without loops

Write a function `distance_matrix(points)` that takes an array of shape
`(n, 2)` holding the coordinates of $n$ points in the plane, and returns the
array `D` of shape `(n, n)` whose entry `D[i, j]` is the Euclidean distance
between point $i$ and point $j$.

Use `np.newaxis` (or `None`) and broadcasting. Do not write a `for` or `while`
loop, do not use a list comprehension, and do not use `scipy`.

For example, for the points `[[0, 0], [3, 0], [0, 4]]` the result is

```
[[0. 3. 4.]
 [3. 0. 5.]
 [4. 5. 0.]]
```


In [2]:
import numpy as np


def distance_matrix(points):
    d = points[:, np.newaxis, :] - points[np.newaxis, :, :]
    return np.sqrt((d ** 2).sum(axis=2))
    # np.linalg.norm(d, axis=2) is equally good.


# Check against the example in the question.
p = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0]])
D = distance_matrix(p)
print(D)
print("shape    :", D.shape)
print("symmetric:", np.allclose(D, D.T))
print("zero diag:", np.allclose(np.diag(D), 0.0))

# The unit square with its centre: every corner is sqrt(0.5) from the middle.
q = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.5, 0.5]])
print(np.round(distance_matrix(q), 4))

[[0. 3. 4.]
 [3. 0. 5.]
 [4. 5. 0.]]
shape    : (3, 3)
symmetric: True
zero diag: True
[[0.     1.     1.4142 1.     0.7071]
 [1.     0.     1.     1.4142 0.7071]
 [1.4142 1.     0.     1.     0.7071]
 [1.     1.4142 1.     0.     0.7071]
 [0.7071 0.7071 0.7071 0.7071 0.    ]]


## Question 3. Counting grid points inside a circle

Write a function `circle_grid(n, r)` that does the following.

Build a square grid of `n * n` points by taking `np.linspace(-1, 1, n)` along
both axes and passing it to `np.meshgrid`. Then return the **number** of grid
points $(x, y)$ that satisfy

$$x^2 + y^2 \leq r^2$$

Use a boolean mask to do the counting. Do not write a `for` or `while` loop and
do not use a list comprehension.

For example, `circle_grid(3, 1.0)` returns `5`: of the nine grid points, the
four corners lie outside the unit circle and the remaining five lie on or
inside it. Note that the boundary counts, so the comparison is $\leq$ and not
$<$.


In [3]:
import numpy as np


def circle_grid(n, r):
    t = np.linspace(-1, 1, n)
    x, y = np.meshgrid(t, t)
    return int((x * x + y * y <= r * r).sum())


# Check against the example in the question, and a few more.
print("circle_grid(3, 1.0)   =", circle_grid(3, 1.0), " expected 5")
print("circle_grid(2, 1.0)   =", circle_grid(2, 1.0), " expected 0")
print("circle_grid(3, 0.5)   =", circle_grid(3, 0.5), " expected 1")
print("circle_grid(5, 1.2)   =", circle_grid(5, 1.2), " expected 21")
print("circle_grid(100, 0.75) =", circle_grid(100, 0.75), " expected 4344")

# Note the two traps: the test is <= and not <, so points exactly on the circle
# count; and it compares against r*r, not against r.

circle_grid(3, 1.0)   = 5  expected 5
circle_grid(2, 1.0)   = 0  expected 0
circle_grid(3, 0.5)   = 1  expected 1
circle_grid(5, 1.2)   = 21  expected 21
circle_grid(100, 0.75) = 4344  expected 4344


## Question 4. Interleave two arrays

Write a function `interleave(a, b)` that takes two one dimensional NumPy arrays
of the same length $n$ and returns a single one dimensional array of length
`2 * n` whose entries alternate between the two inputs, starting with `a`:

```
a[0], b[0], a[1], b[1], ..., a[n-1], b[n-1]
```

Build it with `np.stack` (or `np.column_stack`) followed by a reshape or a
ravel. Do not write a `for` or `while` loop and do not use a list
comprehension.

For example, `interleave(np.array([1, 2, 3]), np.array([4, 5, 6]))` returns
`array([1, 4, 2, 5, 3, 6])`.


In [4]:
import numpy as np


def interleave(a, b):
    return np.stack((a, b), axis=1).ravel()
    # np.column_stack((a, b)).reshape(-1) is the same thing.


# Check against the example in the question.
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
r = interleave(a, b)
print("result:", r, " expected [1 4 2 5 3 6]")
print("shape :", r.shape)

# The shape of the intermediate is what makes this work: stacking on axis=1
# puts each a[i] next to its b[i], and ravel then reads the pairs in order.
print("np.stack((a, b), axis=1) =\n", np.stack((a, b), axis=1))
print("np.stack((a, b))         =\n", np.stack((a, b)))

result: [1 4 2 5 3 6]  expected [1 4 2 5 3 6]
shape : (6,)
np.stack((a, b), axis=1) =
 [[1 4]
 [2 5]
 [3 6]]
np.stack((a, b))         =
 [[1 2 3]
 [4 5 6]]


## Question 5. Least squares straight line fit

Write a function `fit_line(x, y)` that fits the straight line

$$y = m x + c$$

to the data in the one dimensional arrays `x` and `y` in the least squares
sense, and returns the tuple `(m, c)` of two numbers, slope first.

Build the design matrix whose first column is `x` and whose second column is
`np.ones_like(x)`, and pass it to `np.linalg.lstsq` with `rcond=None`. Remember
that `lstsq` returns a tuple whose **first** element is the solution vector.

Do not use `np.polyfit`, `scipy` or any other fitting routine.

For example, for `x = [0, 1, 2, 3, 4]` and `y = [2, 5, 8, 11, 14]` the answer is
`(3.0, 2.0)`.


In [5]:
import numpy as np


def fit_line(x, y):
    A = np.array([x, np.ones_like(x)]).T
    m, c = np.linalg.lstsq(A, y, rcond=None)[0]
    return m, c


# Check against the example in the question: data exactly on y = 3x + 2.
x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = 3.0 * x + 2.0
print("fit_line on y = 3x + 2 -> m = %.6f, c = %.6f  (expected 3 and 2)"
      % fit_line(x, y))

# Data that is not exactly on a line.
x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([2.0, 3.0, 5.0, 6.0])
m, c = fit_line(x, y)
print("m = %.4f, c = %.4f  expected 1.4 and 0.5" % (m, c))

# The column of ones is what allows a non-zero intercept; without it the line
# is forced through the origin.
print("design matrix:\n", np.array([x, np.ones_like(x)]).T)

fit_line on y = 3x + 2 -> m = 3.000000, c = 2.000000  (expected 3 and 2)
m = 1.4000, c = 0.5000  expected 1.4 and 0.5
design matrix:
 [[1. 1.]
 [2. 1.]
 [3. 1.]
 [4. 1.]]
